# RSNA Knee MRI — Clean Self-Supervised Learning Pipeline

Clean rebuild of the SSL preprocessing and training pipeline.

Validated reference targets:

- Unlabelled studies: 4,349
- Usable MRI series: 24,035
- Indexed slices: 808,550
- Valid 2.5D triplets: 760,480
- Train studies: 3,914
- Validation studies: 435
- Train series: 21,640
- Validation series: 2,395
- Balanced train samples / epoch: 86,560
- Fixed validation samples: 9,580

In [1]:
# ============================================================
# SIMPLE RESUME CELL — load saved Version 2 data
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import pydicom


# ------------------------------------------------------------
# Saved notebook output
# ------------------------------------------------------------

PREP_ROOT = Path(
    "/kaggle/input/notebooks/elliotyang37/"
    "final-version/rsna_ssl_clean"
)


# ------------------------------------------------------------
# Load the important saved tables
# ------------------------------------------------------------

ssl_series_df = pd.read_csv(
    PREP_ROOT / "indexes/step02_ssl_series_inventory.csv"
)

ssl_slice_index = pd.read_parquet(
    PREP_ROOT / "indexes/step03_ssl_slice_index.parquet"
)

ssl_triplets_split = pd.read_parquet(
    PREP_ROOT / "splits/step05_triplets_with_split.parquet"
)

series_percentiles = pd.read_parquet(
    PREP_ROOT / "indexes/step06_series_percentiles.parquet"
)


print("Loaded:")
print("Series:", len(ssl_series_df))
print("Slices:", len(ssl_slice_index))
print("Triplets:", len(ssl_triplets_split))

print()
print("Percentile cache:")
print(series_percentiles["Valid"].value_counts())

Loaded:
Series: 24035
Slices: 808550
Triplets: 760480

Percentile cache:
Valid
True     24033
False        2
Name: count, dtype: int64


In [ ]:
from pathlib import Path
import os
import json
import random
import platform
import sys

import numpy as np
import pandas as pd

import torch

import pydicom

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("pydicom:", pydicom.__version__)

print()
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# CLEAN STEP 1
# Global configuration
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# Image preprocessing
# ------------------------------------------------------------

IMAGE_SIZE = 224

LOW_PERCENTILE = 1.0
HIGH_PERCENTILE = 99.0

TRIPLETS_PER_SERIES = 4


# ------------------------------------------------------------
# Validated reference targets
# ------------------------------------------------------------

EXPECTED = {
    "unlabelled_studies": 4349,
    "usable_series": 24035,
    "indexed_slices": 808550,
    "valid_triplets": 760480,

    "train_studies": 3914,
    "val_studies": 435,

    "train_series": 21640,
    "val_series": 2395,

    "train_samples_per_epoch": 86560,
    "val_samples": 9580,
}


# ------------------------------------------------------------
# Kaggle paths
# ------------------------------------------------------------

KAGGLE_INPUT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/rsna_ssl_clean")

CHECKPOINT_DIR = WORK_DIR / "checkpoints"
INDEX_DIR = WORK_DIR / "indexes"
SPLIT_DIR = WORK_DIR / "splits"
OUTPUT_DIR = WORK_DIR / "outputs"

for directory in [
    WORK_DIR,
    CHECKPOINT_DIR,
    INDEX_DIR,
    SPLIT_DIR,
    OUTPUT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


print("Seed:", SEED)
print("Image size:", IMAGE_SIZE)
print("Triplets per series:", TRIPLETS_PER_SERIES)

print()
print("Working directory:")
print(WORK_DIR)

print()
print("Checkpoint directory:")
print(CHECKPOINT_DIR)

In [ ]:
expected_df = pd.DataFrame(
    EXPECTED.items(),
    columns=["Metric", "Expected"]
)

expected_df

assert EXPECTED["train_series"] * TRIPLETS_PER_SERIES == EXPECTED["train_samples_per_epoch"]
assert EXPECTED["val_series"] * TRIPLETS_PER_SERIES == EXPECTED["val_samples"]

print("Balanced sampling counts are internally consistent.")

In [ ]:
# ============================================================
# Inspect Kaggle input datasets
# ============================================================

input_folders = sorted(
    [p for p in KAGGLE_INPUT.iterdir() if p.is_dir()]
)

print(f"Datasets attached: {len(input_folders)}")
print()

for folder in input_folders:
    print(folder.name)

In [ ]:
for root in input_folders:

    print("=" * 80)
    print(root)
    print("=" * 80)

    children = sorted(root.iterdir())

    for child in children[:30]:

        if child.is_dir():
            print("[DIR ]", child.name)
        else:
            print("[FILE]", child.name)

    if len(children) > 30:
        print(f"... plus {len(children) - 30} more items")

    print()

In [ ]:
step1_checkpoint = {
    "step": 1,
    "name": "clean_setup",
    "seed": SEED,
    "image_size": IMAGE_SIZE,
    "low_percentile": LOW_PERCENTILE,
    "high_percentile": HIGH_PERCENTILE,
    "triplets_per_series": TRIPLETS_PER_SERIES,
    "expected": EXPECTED,
    "python_version": sys.version,
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
}

if torch.cuda.is_available():
    step1_checkpoint["gpu"] = torch.cuda.get_device_name(0)

checkpoint_path = CHECKPOINT_DIR / "step01_setup.json"

with open(checkpoint_path, "w") as f:
    json.dump(step1_checkpoint, f, indent=2)

print("Saved:")
print(checkpoint_path)

In [ ]:
# ============================================================
# CLEAN STEP 2A
# Inspect raw competition data layout
# ============================================================

print("Kaggle input folders:\n")

for root in sorted(KAGGLE_INPUT.iterdir()):

    if not root.is_dir():
        continue

    print("=" * 80)
    print("ROOT:", root)
    print("=" * 80)

    children = sorted(root.iterdir())

    for child in children:
        if child.is_dir():
            print("[DIR ]", child.name)
        else:
            print("[FILE]", child.name)

    print()

# Find metadata / label CSV files only

csv_files = sorted(KAGGLE_INPUT.rglob("*.csv"))

print(f"CSV files found: {len(csv_files)}\n")

for file in csv_files:
    print(file)

In [ ]:
# ============================================================
# CLEAN STEP 2B
# Inspect competition structure
# ============================================================

DATA_ROOT = Path(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)

print("DATA_ROOT exists:", DATA_ROOT.exists())
print()

for item in sorted(DATA_ROOT.iterdir()):
    if item.is_dir():
        print("[DIR ]", item.name)
    else:
        print("[FILE]", item.name)

train_df = pd.read_csv(DATA_ROOT / "train.csv")
train_series_df = pd.read_csv(DATA_ROOT / "train_series.csv")

print("train.csv shape:")
print(train_df.shape)

print("\ntrain.csv columns:")
print(train_df.columns.tolist())

print("\nFirst rows:")
display(train_df.head())

print("\n" + "=" * 80)

print("\ntrain_series.csv shape:")
print(train_series_df.shape)

print("\ntrain_series.csv columns:")
print(train_series_df.columns.tolist())

print("\nFirst rows:")
display(train_series_df.head())

# Basic study / series counts

print("Unique studies in train.csv:")
for col in train_df.columns:
    if "study" in col.lower():
        print(col, "->", train_df[col].nunique(dropna=True))

print("\nUnique studies / series in train_series.csv:")
for col in train_series_df.columns:
    if "study" in col.lower() or "series" in col.lower():
        print(col, "->", train_series_df[col].nunique(dropna=True))

In [ ]:
# ============================================================
# CLEAN STEP 2C
# Identify labelled vs unlabelled studies
# ============================================================

LABEL_COLS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

# A study is considered labelled if at least one target is present
has_label = train_df[LABEL_COLS].notna().any(axis=1)

labelled_studies = train_df.loc[
    has_label,
    "StudyInstanceUID"
].unique()

unlabelled_studies = train_df.loc[
    ~has_label,
    "StudyInstanceUID"
].unique()


print("Total studies:", train_df["StudyInstanceUID"].nunique())
print("Labelled studies:", len(labelled_studies))
print("Unlabelled studies:", len(unlabelled_studies))

print()

assert len(unlabelled_studies) == EXPECTED["unlabelled_studies"]

print("✓ Unlabelled study count matches validated target.")

In [ ]:
# ============================================================
# Build SSL series inventory from unlabelled studies only
# ============================================================

ssl_series_df = (
    train_series_df[
        train_series_df["StudyInstanceUID"].isin(unlabelled_studies)
    ]
    .copy()
    .reset_index(drop=True)
)

labelled_series_df = (
    train_series_df[
        train_series_df["StudyInstanceUID"].isin(labelled_studies)
    ]
    .copy()
    .reset_index(drop=True)
)


print("All series:", len(train_series_df))
print("Series from labelled studies:", len(labelled_series_df))
print("SSL / unlabelled series:", len(ssl_series_df))

print()

print(
    "SSL unique studies:",
    ssl_series_df["StudyInstanceUID"].nunique()
)

print(
    "SSL unique series:",
    ssl_series_df["SeriesInstanceUID"].nunique()
)

In [ ]:
TRAIN_SERIES_ROOT = DATA_ROOT / "train_series"

ssl_series_df["SeriesPath"] = ssl_series_df.apply(
    lambda row:
        TRAIN_SERIES_ROOT
        / str(row["StudyInstanceUID"])
        / str(row["SeriesInstanceUID"]),
    axis=1,
)

ssl_series_df["PathExists"] = ssl_series_df["SeriesPath"].map(
    lambda p: p.exists()
)


print("SSL series:", len(ssl_series_df))

print(
    "Series folders found:",
    ssl_series_df["PathExists"].sum()
)

print(
    "Series folders missing:",
    (~ssl_series_df["PathExists"]).sum()
)

In [ ]:
# ============================================================
# Validate Clean Step 2
# ============================================================

assert ssl_series_df["StudyInstanceUID"].nunique() == EXPECTED["unlabelled_studies"]
assert ssl_series_df["SeriesInstanceUID"].nunique() == EXPECTED["usable_series"]
assert ssl_series_df["PathExists"].all()

print("✓ Studies:", EXPECTED["unlabelled_studies"])
print("✓ Series:", EXPECTED["usable_series"])
print("✓ All series folders exist")
print()
print("CLEAN STEP 2 VALIDATED")

# Store paths as strings before saving
ssl_series_save = ssl_series_df.copy()

ssl_series_save["SeriesPath"] = (
    ssl_series_save["SeriesPath"].astype(str)
)

series_checkpoint_path = (
    INDEX_DIR / "step02_ssl_series_inventory.csv"
)

ssl_series_save.to_csv(
    series_checkpoint_path,
    index=False
)

print("Saved:")
print(series_checkpoint_path)

print()
print("Rows saved:", len(ssl_series_save))

check_step2 = pd.read_csv(series_checkpoint_path)

print("Reloaded rows:", len(check_step2))
print(
    "Reloaded studies:",
    check_step2["StudyInstanceUID"].nunique()
)
print(
    "Reloaded series:",
    check_step2["SeriesInstanceUID"].nunique()
)

assert len(check_step2) == 24035
assert check_step2["StudyInstanceUID"].nunique() == 4349

print()
print("✓ Step 2 checkpoint successfully reloaded.")

In [ ]:
# ============================================================
# CLEAN STEP 3A
# DICOM physical slice position
# ============================================================

from tqdm.auto import tqdm


def get_slice_position(ds):
    """
    Return a sortable physical position for a DICOM slice.

    Preferred:
        ImagePositionPatient projected onto the slice normal.

    Fallbacks:
        SliceLocation
        InstanceNumber

    Returns
    -------
    position : float or None
    source   : str
    """

    # --------------------------------------------------------
    # Preferred: true physical position
    # --------------------------------------------------------
    if (
        hasattr(ds, "ImagePositionPatient")
        and hasattr(ds, "ImageOrientationPatient")
    ):
        try:
            ipp = np.asarray(
                ds.ImagePositionPatient,
                dtype=np.float64
            )

            iop = np.asarray(
                ds.ImageOrientationPatient,
                dtype=np.float64
            )

            row_direction = iop[:3]
            col_direction = iop[3:]

            normal = np.cross(
                row_direction,
                col_direction
            )

            position = float(
                np.dot(ipp, normal)
            )

            return position, "ImagePositionPatient"

        except Exception:
            pass

    # --------------------------------------------------------
    # Fallback 1
    # --------------------------------------------------------
    if hasattr(ds, "SliceLocation"):
        try:
            return float(ds.SliceLocation), "SliceLocation"
        except Exception:
            pass

    # --------------------------------------------------------
    # Fallback 2
    # --------------------------------------------------------
    if hasattr(ds, "InstanceNumber"):
        try:
            return float(ds.InstanceNumber), "InstanceNumber"
        except Exception:
            pass

    return None, "Missing"

print("get_slice_position() ready.")

In [ ]:
# ============================================================
# CLEAN STEP 3B
# Prepare slice-index shards
# ============================================================

SLICE_SHARD_DIR = INDEX_DIR / "step03_slice_shards"
SLICE_SHARD_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_SLICE_INDEX = (
    INDEX_DIR / "step03_ssl_slice_index.parquet"
)

SHARD_SIZE = 500

print("Shard directory:")
print(SLICE_SHARD_DIR)

print()
print("Series per shard:", SHARD_SIZE)

print()
print(
    "Approximate number of shards:",
    int(np.ceil(len(ssl_series_df) / SHARD_SIZE))
)

In [ ]:
stop_before_pixels=True

# ============================================================
# CLEAN STEP 3C
# Build physical DICOM slice index
# ============================================================

series_for_indexing = ssl_series_df.reset_index(drop=True)

n_series = len(series_for_indexing)

n_shards = int(
    np.ceil(n_series / SHARD_SIZE)
)


for shard_id in range(n_shards):

    start = shard_id * SHARD_SIZE
    end = min(
        start + SHARD_SIZE,
        n_series
    )

    shard_path = (
        SLICE_SHARD_DIR
        / f"slice_index_{shard_id:03d}.parquet"
    )

    # --------------------------------------------------------
    # Resume support
    # --------------------------------------------------------
    if shard_path.exists():

        print(
            f"Shard {shard_id:03d} already exists "
            f"— skipping."
        )

        continue

    shard_rows = []

    shard_series = series_for_indexing.iloc[start:end]

    for row in tqdm(
        shard_series.itertuples(index=False),
        total=len(shard_series),
        desc=f"Shard {shard_id + 1}/{n_shards}"
    ):

        series_path = Path(row.SeriesPath)

        for file in series_path.glob("*.dcm"):

            try:

                ds = pydicom.dcmread(
                    file,
                    stop_before_pixels=True,
                    force=True
                )

                position, position_source = (
                    get_slice_position(ds)
                )

                # We keep slices even if physical position
                # needed a fallback.
                if position is None:
                    continue

                pixel_spacing = getattr(
                    ds,
                    "PixelSpacing",
                    [np.nan, np.nan]
                )

                try:
                    spacing_y = float(pixel_spacing[0])
                    spacing_x = float(pixel_spacing[1])

                except Exception:
                    spacing_y = np.nan
                    spacing_x = np.nan

                shard_rows.append(
                    {
                        "StudyInstanceUID":
                            row.StudyInstanceUID,

                        "SeriesInstanceUID":
                            row.SeriesInstanceUID,

                        "Anatomical_Plane":
                            row.Anatomical_Plane,

                        "SlicePath":
                            str(file),

                        "SlicePosition":
                            position,

                        "PositionSource":
                            position_source,

                        "InstanceNumber":
                            getattr(
                                ds,
                                "InstanceNumber",
                                np.nan
                            ),

                        "Rows":
                            getattr(
                                ds,
                                "Rows",
                                np.nan
                            ),

                        "Columns":
                            getattr(
                                ds,
                                "Columns",
                                np.nan
                            ),

                        "PixelSpacingY":
                            spacing_y,

                        "PixelSpacingX":
                            spacing_x,
                    }
                )

            except Exception:
                continue

    shard_df = pd.DataFrame(shard_rows)

    # --------------------------------------------------------
    # Sort within each series by physical position
    # --------------------------------------------------------
    if len(shard_df) > 0:

        shard_df = (
            shard_df
            .sort_values(
                [
                    "StudyInstanceUID",
                    "SeriesInstanceUID",
                    "SlicePosition",
                ]
            )
            .reset_index(drop=True)
        )

    shard_df.to_parquet(
        shard_path,
        index=False
    )

    print(
        f"Saved shard {shard_id:03d}: "
        f"{len(shard_df):,} slices"
    )

In [ ]:
# ============================================================
# CLEAN STEP 3D
# Combine slice-index shards
# ============================================================

shard_files = sorted(
    SLICE_SHARD_DIR.glob(
        "slice_index_*.parquet"
    )
)

print("Shard files found:", len(shard_files))
print("Expected shards:", n_shards)

assert len(shard_files) == n_shards

slice_index_parts = []

for file in tqdm(
    shard_files,
    desc="Loading slice-index shards"
):
    slice_index_parts.append(
        pd.read_parquet(file)
    )


ssl_slice_index = pd.concat(
    slice_index_parts,
    ignore_index=True
)

del slice_index_parts


print()
print(
    "Indexed slices:",
    f"{len(ssl_slice_index):,}"
)

print(
    "Studies:",
    ssl_slice_index[
        "StudyInstanceUID"
    ].nunique()
)

print(
    "Series:",
    ssl_slice_index[
        "SeriesInstanceUID"
    ].nunique()
)

In [ ]:
ssl_slice_index["PositionSource"].value_counts(dropna=False)
# ============================================================
# CLEAN STEP 3E
# Finalise and save consolidated slice index
# ============================================================

ssl_slice_index = (
    ssl_slice_index
    .sort_values(
        [
            "StudyInstanceUID",
            "SeriesInstanceUID",
            "SlicePosition",
        ]
    )
    .reset_index(drop=True)
)

# Position of each slice within its physical series
ssl_slice_index["SliceIndex"] = (
    ssl_slice_index
    .groupby("SeriesInstanceUID")
    .cumcount()
)

ssl_slice_index.to_parquet(
    FINAL_SLICE_INDEX,
    index=False
)

print("Saved final slice index:")
print(FINAL_SLICE_INDEX)

print()
print("Rows:", f"{len(ssl_slice_index):,}")
print(
    "Studies:",
    ssl_slice_index["StudyInstanceUID"].nunique()
)
print(
    "Series:",
    ssl_slice_index["SeriesInstanceUID"].nunique()
)

In [ ]:
# ============================================================
# Verify Step 3 checkpoint
# ============================================================

check_step3 = pd.read_parquet(
    FINAL_SLICE_INDEX
)

assert len(check_step3) == EXPECTED["indexed_slices"]

assert (
    check_step3["StudyInstanceUID"].nunique()
    == EXPECTED["unlabelled_studies"]
)

assert (
    check_step3["SeriesInstanceUID"].nunique()
    == EXPECTED["usable_series"]
)

assert (
    check_step3["PositionSource"]
    .eq("ImagePositionPatient")
    .all()
)

print("✓ Indexed slices:", f"{len(check_step3):,}")
print(
    "✓ Studies:",
    check_step3["StudyInstanceUID"].nunique()
)
print(
    "✓ Series:",
    check_step3["SeriesInstanceUID"].nunique()
)
print("✓ All slices use physical ImagePositionPatient")
print()
print("CLEAN STEP 3 VALIDATED")

In [ ]:
# ============================================================
# CLEAN STEP 4A
# Build previous / centre / next triplets
# ============================================================

triplet_source = (
    ssl_slice_index
    .sort_values(
        [
            "StudyInstanceUID",
            "SeriesInstanceUID",
            "SlicePosition",
        ]
    )
    .reset_index(drop=True)
)

grouped = triplet_source.groupby(
    "SeriesInstanceUID",
    sort=False
)


# ------------------------------------------------------------
# Adjacent paths
# ------------------------------------------------------------

triplet_source["PreviousPath"] = grouped[
    "SlicePath"
].shift(1)

triplet_source["NextPath"] = grouped[
    "SlicePath"
].shift(-1)


# ------------------------------------------------------------
# Adjacent physical positions
# ------------------------------------------------------------

triplet_source["PreviousPosition"] = grouped[
    "SlicePosition"
].shift(1)

triplet_source["NextPosition"] = grouped[
    "SlicePosition"
].shift(-1)


# ------------------------------------------------------------
# Keep only slices with both neighbours
# ------------------------------------------------------------

ssl_triplets = (
    triplet_source[
        triplet_source["PreviousPath"].notna()
        &
        triplet_source["NextPath"].notna()
    ]
    .copy()
    .reset_index(drop=True)
)


# Rename centre fields clearly
ssl_triplets = ssl_triplets.rename(
    columns={
        "SlicePath": "CentrePath",
        "SlicePosition": "CentrePosition",
        "SliceIndex": "CentreIndex",
    }
)


print(
    "Valid 2.5D triplets:",
    f"{len(ssl_triplets):,}"
)

print(
    "Studies:",
    ssl_triplets["StudyInstanceUID"].nunique()
)

print(
    "Series:",
    ssl_triplets["SeriesInstanceUID"].nunique()
)

In [ ]:
# ============================================================
# CLEAN STEP 4B
# Triplet QC
# ============================================================

assert len(ssl_triplets) == EXPECTED["valid_triplets"]

assert (
    ssl_triplets["StudyInstanceUID"].nunique()
    == EXPECTED["unlabelled_studies"]
)

assert (
    ssl_triplets["SeriesInstanceUID"].nunique()
    == EXPECTED["usable_series"]
)


# Physical positions must be ordered
physical_order_ok = (
    (
        ssl_triplets["PreviousPosition"]
        < ssl_triplets["CentrePosition"]
    )
    &
    (
        ssl_triplets["CentrePosition"]
        < ssl_triplets["NextPosition"]
    )
)

print(
    "Correct physical ordering:",
    f"{physical_order_ok.mean():.6f}"
)

assert physical_order_ok.all()

print()
print("✓ Triplet count:", f"{len(ssl_triplets):,}")
print("✓ Every triplet stays within one series")
print("✓ Previous < Centre < Next")
print()
print("CLEAN STEP 4 VALIDATED")

In [ ]:
TRIPLET_INDEX_PATH = (
    INDEX_DIR / "step04_ssl_triplets.parquet"
)

ssl_triplets.to_parquet(
    TRIPLET_INDEX_PATH,
    index=False
)

print("Saved:")
print(TRIPLET_INDEX_PATH)

print()
print(
    "Triplets saved:",
    f"{len(ssl_triplets):,}"
)


check_step4 = pd.read_parquet(
    TRIPLET_INDEX_PATH
)

assert len(check_step4) == 760480
assert (
    check_step4["SeriesInstanceUID"].nunique()
    == 24035
)
assert (
    check_step4["StudyInstanceUID"].nunique()
    == 4349
)

print(
    "Reloaded triplets:",
    f"{len(check_step4):,}"
)
print(
    "Reloaded studies:",
    check_step4["StudyInstanceUID"].nunique()
)
print(
    "Reloaded series:",
    check_step4["SeriesInstanceUID"].nunique()
)

print()
print("✓ Step 4 checkpoint successfully reloaded.")

In [ ]:
# ============================================================
# CLEAN STEP 5A
# Reproduce validated study-level split EXACTLY
# ============================================================

RANDOM_SEED = 42
VAL_FRACTION = 0.10


# Important:
# Sort study IDs BEFORE shuffling.
ssl_studies = (
    ssl_slice_index["StudyInstanceUID"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

print("Total SSL studies:", len(ssl_studies))


# Deterministic NumPy generator
rng = np.random.default_rng(RANDOM_SEED)

shuffled_studies = ssl_studies.to_numpy().copy()

rng.shuffle(shuffled_studies)


# Validation size
n_val = round(
    len(shuffled_studies) * VAL_FRACTION
)


# First 10% -> validation
val_studies = set(
    shuffled_studies[:n_val]
)

# Remaining -> training
train_studies = set(
    shuffled_studies[n_val:]
)


print("Train studies:", len(train_studies))
print("Validation studies:", len(val_studies))

print(
    "Study overlap:",
    len(train_studies & val_studies)
)

In [ ]:
# ============================================================
# CLEAN STEP 5B
# Assign every series to its study-level split
# ============================================================

ssl_series_split = ssl_series_df.copy()

ssl_series_split["SSL_Split"] = np.where(
    ssl_series_split["StudyInstanceUID"].isin(train_studies),
    "train",
    "val"
)


print("Series by split:")
print(
    ssl_series_split["SSL_Split"]
    .value_counts()
)

print()

print("Studies by split:")
print(
    ssl_series_split
    .groupby("SSL_Split")["StudyInstanceUID"]
    .nunique()
)

In [ ]:
# ============================================================
# CLEAN STEP 5C
# Validate study-level split
# ============================================================

assert len(train_studies) == EXPECTED["train_studies"]
assert len(val_studies) == EXPECTED["val_studies"]

assert len(train_studies & val_studies) == 0

series_counts = (
    ssl_series_split["SSL_Split"]
    .value_counts()
)

assert series_counts["train"] == EXPECTED["train_series"]
assert series_counts["val"] == EXPECTED["val_series"]


print("✓ Train studies:", len(train_studies))
print("✓ Val studies:", len(val_studies))

print("✓ Train series:", series_counts["train"])
print("✓ Val series:", series_counts["val"])

print("✓ Study leakage: 0")

print()
print("CLEAN STEP 5 VALIDATED")

In [ ]:
# ============================================================
# Save study split
# ============================================================

study_split_df = pd.DataFrame({
    "StudyInstanceUID":
        list(train_studies) + list(val_studies),

    "SSL_Split":
        ["train"] * len(train_studies)
        + ["val"] * len(val_studies)
})

study_split_df = (
    study_split_df
    .sort_values("StudyInstanceUID")
    .reset_index(drop=True)
)

STUDY_SPLIT_PATH = (
    SPLIT_DIR / "step05_study_split.csv"
)

study_split_df.to_csv(
    STUDY_SPLIT_PATH,
    index=False
)


# ============================================================
# Save series split
# ============================================================

SERIES_SPLIT_PATH = (
    SPLIT_DIR / "step05_series_split.csv"
)

ssl_series_split_save = ssl_series_split.copy()

ssl_series_split_save["SeriesPath"] = (
    ssl_series_split_save["SeriesPath"]
    .astype(str)
)

ssl_series_split_save.to_csv(
    SERIES_SPLIT_PATH,
    index=False
)


print("Saved study split:")
print(STUDY_SPLIT_PATH)

print()

print("Saved series split:")
print(SERIES_SPLIT_PATH)

In [ ]:
# ============================================================
# CLEAN STEP 5E
# Attach study split to triplet index
# ============================================================

ssl_triplets_split = ssl_triplets.copy()

ssl_triplets_split["SSL_Split"] = np.where(
    ssl_triplets_split["StudyInstanceUID"].isin(train_studies),
    "train",
    "val"
)


print("Triplets by split:")

print(
    ssl_triplets_split["SSL_Split"]
    .value_counts()
)

print()

print("Studies represented:")

print(
    ssl_triplets_split
    .groupby("SSL_Split")["StudyInstanceUID"]
    .nunique()
)

print()

print("Series represented:")

print(
    ssl_triplets_split
    .groupby("SSL_Split")["SeriesInstanceUID"]
    .nunique()
)

In [ ]:
TRIPLET_SPLIT_PATH = (
    SPLIT_DIR / "step05_triplets_with_split.parquet"
)

ssl_triplets_split.to_parquet(
    TRIPLET_SPLIT_PATH,
    index=False
)

# Reload verification
check_step5 = pd.read_parquet(
    TRIPLET_SPLIT_PATH
)

assert len(check_step5) == EXPECTED["valid_triplets"]

assert (
    check_step5[
        check_step5["SSL_Split"] == "train"
    ]["StudyInstanceUID"].nunique()
    == EXPECTED["train_studies"]
)

assert (
    check_step5[
        check_step5["SSL_Split"] == "val"
    ]["StudyInstanceUID"].nunique()
    == EXPECTED["val_studies"]
)

print("Reloaded triplets:", f"{len(check_step5):,}")

print()
print("✓ Step 5 checkpoints successfully reloaded.")

In [ ]:
# ============================================================
# CLEAN STEP 6A
# Inspect in-plane pixel spacing
# ============================================================

spacing_qc = (
    ssl_slice_index[
        ["SeriesInstanceUID", "PixelSpacingY", "PixelSpacingX"]
    ]
    .drop_duplicates("SeriesInstanceUID")
    .copy()
)

print("Series with spacing metadata:", len(spacing_qc))

print("\nPixelSpacingY summary:")
print(spacing_qc["PixelSpacingY"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
))

print("\nPixelSpacingX summary:")
print(spacing_qc["PixelSpacingX"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
))

print("\nMissing Y:", spacing_qc["PixelSpacingY"].isna().sum())
print("Missing X:", spacing_qc["PixelSpacingX"].isna().sum())

In [ ]:
print(
    spacing_qc[
        ["PixelSpacingY", "PixelSpacingX"]
    ]
    .round(4)
    .value_counts()
    .head(20)
)

In [ ]:
# ============================================================
# CLEAN STEP 6B
# MRI preprocessing configuration
# ============================================================

import torch.nn.functional as F


TARGET_SPACING = 0.3125   # mm / pixel
TARGET_SIZE = 224

PERCENTILE_LOW = 1.0
PERCENTILE_HIGH = 99.0


print("Target spacing:", TARGET_SPACING, "mm/pixel")
print("Target image size:", TARGET_SIZE)
print(
    "Intensity percentiles:",
    PERCENTILE_LOW,
    "-",
    PERCENTILE_HIGH
)

In [ ]:
# ============================================================
# CLEAN STEP 6C
# Load one DICOM slice correctly
# ============================================================

def load_dicom_pixels(path):
    """
    Load one MRI DICOM slice.

    Applies:
    - RescaleSlope
    - RescaleIntercept
    - MONOCHROME1 inversion when required

    Returns
    -------
    image : np.ndarray, float32
    spacing_y : float
    spacing_x : float
    """

    ds = pydicom.dcmread(
        str(path),
        force=True
    )

    image = ds.pixel_array.astype(np.float32)


    # --------------------------------------------------------
    # DICOM rescale
    # --------------------------------------------------------

    slope = float(
        getattr(ds, "RescaleSlope", 1.0)
    )

    intercept = float(
        getattr(ds, "RescaleIntercept", 0.0)
    )

    image = image * slope + intercept


    # --------------------------------------------------------
    # MONOCHROME1:
    # high stored value = dark rather than bright
    # --------------------------------------------------------

    if getattr(
        ds,
        "PhotometricInterpretation",
        ""
    ) == "MONOCHROME1":

        image = image.max() + image.min() - image


    # --------------------------------------------------------
    # Physical pixel spacing
    # --------------------------------------------------------

    spacing = ds.PixelSpacing

    spacing_y = float(spacing[0])
    spacing_x = float(spacing[1])


    return (
        image,
        spacing_y,
        spacing_x
    )

In [ ]:
# ============================================================
# CLEAN STEP 6D
# Percentile intensity normalization
# ============================================================

def percentile_normalize(
    image,
    low_value,
    high_value
):
    """
    Normalize an MRI slice using precomputed
    series-level percentile bounds.
    """

    image = np.clip(
        image,
        low_value,
        high_value
    )

    denominator = (
        high_value - low_value
    )

    if denominator <= 1e-8:
        return np.zeros_like(
            image,
            dtype=np.float32
        )

    image = (
        image - low_value
    ) / denominator

    return image.astype(np.float32)

In [ ]:
# ============================================================
# CLEAN STEP 6E
# Physical-spacing-aware resize
# ============================================================

def resize_preserve_physical_fov(
    image,
    spacing_y,
    spacing_x,
    target_size=TARGET_SIZE
):
    """
    Resize an MRI slice so the complete physical field-of-view
    fits inside target_size x target_size.

    Physical dimensions are calculated from:
        pixels × mm/pixel

    This preserves physical aspect ratio and avoids aggressive
    cropping of the knee.
    """

    h, w = image.shape

    # Physical field-of-view in millimetres
    physical_h = h * float(spacing_y)
    physical_w = w * float(spacing_x)

    # Pixels per mm required to fit the largest physical
    # dimension into target_size
    scale = min(
        target_size / physical_h,
        target_size / physical_w
    )

    new_h = max(
        1,
        int(round(physical_h * scale))
    )

    new_w = max(
        1,
        int(round(physical_w * scale))
    )

    # Rounding safety
    new_h = min(new_h, target_size)
    new_w = min(new_w, target_size)

    x = torch.from_numpy(
        image
    ).float()[None, None]

    x = F.interpolate(
        x,
        size=(new_h, new_w),
        mode="bilinear",
        align_corners=False
    )

    return x[0, 0]

In [ ]:
# ============================================================
# CLEAN STEP 6F
# Centre crop / pad
# ============================================================

def centre_pad_to_target(
    image,
    target_size=TARGET_SIZE
):
    """
    Centre-pad a resized MRI image to target_size x target_size.

    No meaningful anatomy should be cropped.
    """

    h, w = image.shape

    # Safety only — normally unnecessary because Step 6E
    # already guarantees h,w <= target_size
    if h > target_size:
        top = (h - target_size) // 2
        image = image[
            top:top + target_size,
            :
        ]

    h, w = image.shape

    if w > target_size:
        left = (w - target_size) // 2
        image = image[
            :,
            left:left + target_size
        ]

    h, w = image.shape

    pad_h = target_size - h
    pad_w = target_size - w

    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top

    pad_left = pad_w // 2
    pad_right = pad_w - pad_left

    image = F.pad(
        image,
        (
            pad_left,
            pad_right,
            pad_top,
            pad_bottom
        ),
        mode="constant",
        value=0.0
    )

    return image

In [ ]:
# ============================================================
# CLEAN STEP 6G
# Compute series-level intensity bounds
# ============================================================

def get_series_percentile_bounds(
    slice_paths,
    low=PERCENTILE_LOW,
    high=PERCENTILE_HIGH
):

    pixels = []

    for path in slice_paths:

        image, _, _ = load_dicom_pixels(
            path
        )

        pixels.append(
            image.reshape(-1)
        )


    pixels = np.concatenate(pixels)


    low_value = float(
        np.percentile(
            pixels,
            low
        )
    )

    high_value = float(
        np.percentile(
            pixels,
            high
        )
    )


    return low_value, high_value

In [ ]:
# ============================================================
# CLEAN STEP 6H
# Complete single-slice preprocessing
# ============================================================

def preprocess_slice(
    path,
    low_value,
    high_value
):

    image, spacing_y, spacing_x = (
        load_dicom_pixels(path)
    )

    # Series-level 1st–99th percentile normalization
    image = percentile_normalize(
        image,
        low_value,
        high_value
    )

    # Preserve complete physical FOV
    image = resize_preserve_physical_fov(
        image,
        spacing_y,
        spacing_x
    )

    # Pad to exactly 224 x 224
    image = centre_pad_to_target(
        image
    )

    return image

In [ ]:
# ============================================================
# CLEAN STEP 6I
# Build 3-channel 2.5D MRI sample
# ============================================================

def preprocess_triplet(
    previous_path,
    centre_path,
    next_path,
    low_value,
    high_value
):

    previous = preprocess_slice(
        previous_path,
        low_value,
        high_value
    )

    centre = preprocess_slice(
        centre_path,
        low_value,
        high_value
    )

    next_slice = preprocess_slice(
        next_path,
        low_value,
        high_value
    )


    triplet = torch.stack(
        [
            previous,
            centre,
            next_slice
        ],
        dim=0
    )


    return triplet

In [ ]:
# ============================================================
# CLEAN STEP 6J
# Preprocessing QC on one real MRI series
# ============================================================

test_triplet = ssl_triplets_split.iloc[0]

test_series_uid = (
    test_triplet["SeriesInstanceUID"]
)


series_slices = (
    ssl_slice_index[
        ssl_slice_index["SeriesInstanceUID"]
        == test_series_uid
    ]
    .sort_values("SlicePosition")
)


print(
    "Test series:",
    test_series_uid
)

print(
    "Slices in series:",
    len(series_slices)
)


low_value, high_value = (
    get_series_percentile_bounds(
        series_slices["SlicePath"].tolist()
    )
)


print(
    "Series P1:",
    low_value
)

print(
    "Series P99:",
    high_value
)


sample_25d = preprocess_triplet(
    test_triplet["PreviousPath"],
    test_triplet["CentrePath"],
    test_triplet["NextPath"],
    low_value,
    high_value
)


print()
print(
    "2.5D shape:",
    sample_25d.shape
)

print(
    "Minimum:",
    sample_25d.min().item()
)

print(
    "Maximum:",
    sample_25d.max().item()
)

print(
    "Mean:",
    sample_25d.mean().item()
)

In [ ]:
assert sample_25d.shape == (3, 224, 224)

assert torch.isfinite(
    sample_25d
).all()

assert sample_25d.min() >= 0

assert sample_25d.max() <= 1

print()
print("✓ Shape correct")
print("✓ No NaN / Inf")
print("✓ Intensity range valid")
print()
print("CLEAN STEP 6 PREPROCESSING QC PASSED")

In [ ]:
# ============================================================
# CLEAN STEP 6K
# Visual QC of preprocessed 2.5D triplet
# ============================================================

import matplotlib.pyplot as plt


fig, axes = plt.subplots(
    1,
    3,
    figsize=(12, 4)
)

titles = [
    "Previous",
    "Centre",
    "Next"
]

for i, ax in enumerate(axes):

    ax.imshow(
        sample_25d[i].cpu().numpy(),
        cmap="gray"
    )

    ax.set_title(titles[i])
    ax.axis("off")


plt.suptitle(
    f"Preprocessed 2.5D Triplet — "
    f"{test_triplet['Anatomical_Plane']}"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Corrected physical-FOV QC
# ============================================================

centre_image, sy, sx = load_dicom_pixels(
    test_triplet["CentrePath"]
)

original_h, original_w = centre_image.shape

physical_h = original_h * sy
physical_w = original_w * sx

scale = min(
    TARGET_SIZE / physical_h,
    TARGET_SIZE / physical_w
)

resized_h = round(
    physical_h * scale
)

resized_w = round(
    physical_w * scale
)


print("Original shape:", centre_image.shape)

print(
    "Original spacing:",
    sy,
    sx
)

print(
    "Physical FOV:",
    round(physical_h, 2),
    "x",
    round(physical_w, 2),
    "mm"
)

print(
    "Physical-fit resized size:",
    resized_h,
    "x",
    resized_w
)

print(
    "Final tensor:",
    sample_25d.shape
)

In [ ]:
# ============================================================
# CLEAN STEP 6L
# Visual QC across Sagittal / Coronal / Axial
# ============================================================

planes = ["Sagittal", "Coronal", "Axial"]

qc_samples = {}

for plane in planes:

    row = (
        ssl_triplets_split[
            ssl_triplets_split["Anatomical_Plane"] == plane
        ]
        .iloc[0]
    )

    series_uid = row["SeriesInstanceUID"]

    series_slices = (
        ssl_slice_index[
            ssl_slice_index["SeriesInstanceUID"] == series_uid
        ]
        .sort_values("SlicePosition")
    )

    low_value, high_value = get_series_percentile_bounds(
        series_slices["SlicePath"].tolist()
    )

    sample = preprocess_triplet(
        row["PreviousPath"],
        row["CentrePath"],
        row["NextPath"],
        low_value,
        high_value
    )

    qc_samples[plane] = sample

    print(
        plane,
        "->",
        sample.shape,
        "range:",
        float(sample.min()),
        "to",
        float(sample.max())
    )

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    1,
    3,
    figsize=(12, 4)
)

for ax, plane in zip(axes, planes):

    ax.imshow(
        qc_samples[plane][1].cpu().numpy(),
        cmap="gray"
    )

    ax.set_title(plane)
    ax.axis("off")

plt.suptitle(
    "Clean Step 6 — Preprocessing QC Across MRI Planes"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CLEAN STEP 6M
# Save validated preprocessing configuration
# ============================================================

step6_checkpoint = {
    "step": 6,
    "name": "validated_mri_preprocessing",

    "target_size": TARGET_SIZE,

    "normalization": {
        "type": "series_level_percentile",
        "low_percentile": PERCENTILE_LOW,
        "high_percentile": PERCENTILE_HIGH,
        "output_range": [0.0, 1.0],
    },

    "geometry": {
        "method": "physical_fov_aware_resize",
        "uses_pixel_spacing": True,
        "preserve_complete_fov": True,
        "final_size": [224, 224],
        "padding": "centre_constant_zero",
    },

    "channels": [
        "previous_slice",
        "centre_slice",
        "next_slice",
    ],

    "qc": {
        "sagittal_passed": True,
        "coronal_passed": True,
        "axial_passed": True,
    },
}

STEP6_CONFIG_PATH = (
    CHECKPOINT_DIR / "step06_preprocessing.json"
)

with open(STEP6_CONFIG_PATH, "w") as f:
    json.dump(
        step6_checkpoint,
        f,
        indent=2
    )

print("Saved:")
print(STEP6_CONFIG_PATH)

In [ ]:
# ============================================================
# CLEAN STEP 6N
# Prepare resumable percentile cache
# ============================================================

PERCENTILE_SHARD_DIR = (
    INDEX_DIR / "step06_percentile_shards"
)

PERCENTILE_SHARD_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PERCENTILE_CACHE_PATH = (
    INDEX_DIR / "step06_series_percentiles.parquet"
)

PERCENTILE_SHARD_SIZE = 100


series_ids = (
    ssl_slice_index[
        "SeriesInstanceUID"
    ]
    .drop_duplicates()
    .tolist()
)

n_percentile_shards = int(
    np.ceil(
        len(series_ids)
        / PERCENTILE_SHARD_SIZE
    )
)

print("Series:", len(series_ids))
print(
    "Series per shard:",
    PERCENTILE_SHARD_SIZE
)
print(
    "Number of shards:",
    n_percentile_shards
)

In [ ]:
# ============================================================
# CLEAN STEP 6O
# Cache series-level P1 / P99
# ============================================================

series_paths_lookup = (
    ssl_slice_index
    .groupby("SeriesInstanceUID")["SlicePath"]
    .apply(list)
    .to_dict()
)


for shard_id in range(n_percentile_shards):

    start = shard_id * PERCENTILE_SHARD_SIZE

    end = min(
        start + PERCENTILE_SHARD_SIZE,
        len(series_ids)
    )

    shard_path = (
        PERCENTILE_SHARD_DIR
        / f"percentiles_{shard_id:03d}.parquet"
    )

    # Resume support
    if shard_path.exists():

        print(
            f"Shard {shard_id:03d} exists — skipping."
        )

        continue


    rows = []

    current_ids = series_ids[start:end]


    for series_uid in tqdm(
        current_ids,
        desc=(
            f"Percentiles "
            f"{shard_id + 1}/"
            f"{n_percentile_shards}"
        )
    ):

        slice_paths = (
            series_paths_lookup[series_uid]
        )

        try:

            p1, p99 = (
                get_series_percentile_bounds(
                    slice_paths
                )
            )

            rows.append(
                {
                    "SeriesInstanceUID":
                        series_uid,

                    "P1":
                        p1,

                    "P99":
                        p99,

                    "NumSlices":
                        len(slice_paths),

                    "Valid":
                        bool(p99 > p1),
                }
            )

        except Exception:

            rows.append(
                {
                    "SeriesInstanceUID":
                        series_uid,

                    "P1":
                        np.nan,

                    "P99":
                        np.nan,

                    "NumSlices":
                        len(slice_paths),

                    "Valid":
                        False,
                }
            )


    shard_df = pd.DataFrame(rows)

    shard_df.to_parquet(
        shard_path,
        index=False
    )

    print(
        f"Saved shard {shard_id:03d}: "
        f"{len(shard_df)} series"
    )

In [ ]:
# ============================================================
# CLEAN STEP 6Q
# Inspect invalid percentile series
# ============================================================

invalid_percentiles = (
    series_percentiles[
        ~series_percentiles["Valid"]
    ]
    .copy()
)

display(invalid_percentiles)

invalid_info = (
    invalid_percentiles
    .merge(
        ssl_series_df[
            [
                "StudyInstanceUID",
                "SeriesInstanceUID",
                "Anatomical_Plane",
            ]
        ],
        on="SeriesInstanceUID",
        how="left"
    )
    .merge(
        ssl_triplets_split[
            [
                "SeriesInstanceUID",
                "SSL_Split"
            ]
        ]
        .drop_duplicates("SeriesInstanceUID"),
        on="SeriesInstanceUID",
        how="left"
    )
)

display(invalid_info)

In [ ]:
# ============================================================
# CLEAN STEP 6R
# Pixel-level diagnostic for invalid series
# ============================================================

def diagnostic_load_pixels(path):

    ds = pydicom.dcmread(
        str(path),
        force=True
    )

    image = ds.pixel_array.astype(np.float32)

    slope = float(
        getattr(ds, "RescaleSlope", 1.0)
    )

    intercept = float(
        getattr(ds, "RescaleIntercept", 0.0)
    )

    image = image * slope + intercept

    if getattr(
        ds,
        "PhotometricInterpretation",
        ""
    ) == "MONOCHROME1":

        image = (
            image.max()
            + image.min()
            - image
        )

    return image

for series_uid in invalid_percentiles["SeriesInstanceUID"]:

    print("=" * 80)
    print("Series:", series_uid)

    series_rows = (
        ssl_slice_index[
            ssl_slice_index["SeriesInstanceUID"]
            == series_uid
        ]
        .sort_values("SlicePosition")
    )

    print("Slices:", len(series_rows))

    all_pixels = []
    read_errors = 0

    for path in series_rows["SlicePath"]:

        try:
            image = diagnostic_load_pixels(path)

            if np.isfinite(image).all():
                all_pixels.append(
                    image.reshape(-1)
                )
            else:
                print("Non-finite pixels:", path)

        except Exception as e:
            read_errors += 1
            print(
                "Read error:",
                Path(path).name,
                "->",
                type(e).__name__,
                str(e)[:150]
            )

    print("Read errors:", read_errors)

    if all_pixels:

        pixels = np.concatenate(all_pixels)

        print("Pixel min:", float(pixels.min()))
        print("Pixel max:", float(pixels.max()))

        print(
            "P1:",
            float(np.percentile(pixels, 1))
        )

        print(
            "P99:",
            float(np.percentile(pixels, 99))
        )

        print(
            "Pixel std:",
            float(pixels.std())
        )

    print()

In [ ]:
# ============================================================
# CLEAN STEP 6S
# Identify unreadable DICOM slices
# ============================================================

bad_slice_rows = []

for series_uid in invalid_percentiles["SeriesInstanceUID"]:

    series_rows = (
        ssl_slice_index[
            ssl_slice_index["SeriesInstanceUID"] == series_uid
        ]
        .sort_values("SlicePosition")
    )

    for row in series_rows.itertuples(index=False):

        try:
            _ = diagnostic_load_pixels(row.SlicePath)

        except Exception as e:

            bad_slice_rows.append(
                {
                    "SeriesInstanceUID": series_uid,
                    "SlicePath": row.SlicePath,
                    "Error": str(e),
                }
            )


bad_slices_df = pd.DataFrame(bad_slice_rows)

display(bad_slices_df)

print()
print("Corrupted slices:", len(bad_slices_df))

In [ ]:
# ============================================================
# CLEAN STEP 6T
# Repair percentile cache
# ============================================================

for series_uid in invalid_percentiles["SeriesInstanceUID"]:

    series_rows = (
        ssl_slice_index[
            ssl_slice_index["SeriesInstanceUID"] == series_uid
        ]
        .sort_values("SlicePosition")
    )

    pixels = []

    for path in series_rows["SlicePath"]:

        try:
            image = diagnostic_load_pixels(path)
            pixels.append(image.reshape(-1))

        except Exception:
            continue

    pixels = np.concatenate(pixels)

    p1 = float(np.percentile(pixels, 1))
    p99 = float(np.percentile(pixels, 99))

    mask = (
        series_percentiles["SeriesInstanceUID"]
        == series_uid
    )

    series_percentiles.loc[mask, "P1"] = p1
    series_percentiles.loc[mask, "P99"] = p99
    series_percentiles.loc[mask, "Valid"] = p99 > p1


print(series_percentiles["Valid"].value_counts())

In [ ]:
# ============================================================
# CLEAN STEP 6U
# Remove triplets containing corrupted DICOMs
# ============================================================

bad_paths = set(
    bad_slices_df["SlicePath"].astype(str)
)

before = len(ssl_triplets_split)

bad_triplet_mask = (
    ssl_triplets_split["PreviousPath"].astype(str).isin(bad_paths)
    |
    ssl_triplets_split["CentrePath"].astype(str).isin(bad_paths)
    |
    ssl_triplets_split["NextPath"].astype(str).isin(bad_paths)
)

print("Triplets touching corrupted slices:", bad_triplet_mask.sum())

ssl_triplets_clean = (
    ssl_triplets_split[
        ~bad_triplet_mask
    ]
    .copy()
    .reset_index(drop=True)
)

print("Before:", f"{before:,}")
print("After: ", f"{len(ssl_triplets_clean):,}")
print("Removed:", f"{before - len(ssl_triplets_clean):,}")

print(
    "Studies:",
    ssl_triplets_clean["StudyInstanceUID"].nunique()
)

print(
    "Series:",
    ssl_triplets_clean["SeriesInstanceUID"].nunique()
)

print()

print(
    ssl_triplets_clean
    .groupby("SSL_Split")["SeriesInstanceUID"]
    .nunique()
)

In [ ]:
# ============================================================
# CLEAN STEP 6V
# Save FINAL repaired Step 6 artifacts
# ============================================================

from pathlib import Path
import json

WORK_DIR = Path("/kaggle/working/rsna_ssl_clean")

CHECKPOINT_DIR = WORK_DIR / "checkpoints"
INDEX_DIR = WORK_DIR / "indexes"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)


FINAL_PERCENTILE_PATH = (
    INDEX_DIR / "step06_series_percentiles_FINAL.parquet"
)

FINAL_TRIPLETS_PATH = (
    INDEX_DIR / "step06_clean_triplets_FINAL.parquet"
)

BAD_SLICES_PATH = (
    INDEX_DIR / "step06_corrupted_slices.csv"
)


# Repaired percentile cache
series_percentiles.to_parquet(
    FINAL_PERCENTILE_PATH,
    index=False
)

# Triplets with corrupt files removed
ssl_triplets_clean.to_parquet(
    FINAL_TRIPLETS_PATH,
    index=False
)

# Record the two problematic DICOM files
bad_slices_df.to_csv(
    BAD_SLICES_PATH,
    index=False
)


print("Saved:")
print(FINAL_PERCENTILE_PATH)
print(FINAL_TRIPLETS_PATH)
print(BAD_SLICES_PATH)

In [ ]:
step6_final = {
    "step": 6,
    "status": "validated_final",

    "total_series": 24035,
    "valid_percentile_series": 24035,

    "original_triplets": 760480,
    "corrupted_dicom_slices": 2,
    "removed_triplets": 2,
    "final_clean_triplets": 760478,

    "studies": 4349,

    "train_series": 21640,
    "val_series": 2395,

    "train_samples_per_epoch": 86560,
    "val_samples": 9580,

    "preprocessing": {
        "normalization": "series-level P1-P99",
        "geometry": "physical-FOV-aware resize",
        "output_size": [224, 224],
        "channels": [
            "previous",
            "centre",
            "next"
        ]
    }
}


STEP6_FINAL_CHECKPOINT = (
    CHECKPOINT_DIR / "step06_FINAL.json"
)

with open(
    STEP6_FINAL_CHECKPOINT,
    "w"
) as f:
    json.dump(
        step6_final,
        f,
        indent=2
    )


print()
print("Saved:")
print(STEP6_FINAL_CHECKPOINT)

In [ ]:
# ============================================================
# CLEAN STEP 7A
# MRI-safe SSL augmentation
# ============================================================

import random
import torch
import torchvision.transforms.functional as TF

from torchvision.transforms import InterpolationMode


def mri_ssl_augment(x):
    """
    MRI-safe augmentation for a 2.5D triplet.

    Input:
        x: torch.Tensor [3, 224, 224], range [0, 1]

    Important:
        Spatial transformations are applied jointly to all
        three channels so previous/centre/next remain aligned.
    """

    x = x.clone()

    # --------------------------------------------------------
    # 1. Small spatial affine transformation
    # --------------------------------------------------------

    if random.random() < 0.8:

        angle = random.uniform(-8.0, 8.0)

        translate_x = random.uniform(-0.04, 0.04)
        translate_y = random.uniform(-0.04, 0.04)

        h, w = x.shape[-2:]

        translate = [
            int(round(translate_x * w)),
            int(round(translate_y * h)),
        ]

        scale = random.uniform(0.95, 1.05)

        x = TF.affine(
            x,
            angle=angle,
            translate=translate,
            scale=scale,
            shear=[0.0, 0.0],
            interpolation=InterpolationMode.BILINEAR,
            fill=0.0,
        )


    # --------------------------------------------------------
    # 2. Mild intensity scaling
    # --------------------------------------------------------

    if random.random() < 0.8:

        intensity_scale = random.uniform(
            0.90,
            1.10
        )

        x = x * intensity_scale


    # --------------------------------------------------------
    # 3. Mild intensity shift
    # --------------------------------------------------------

    if random.random() < 0.5:

        intensity_shift = random.uniform(
            -0.05,
            0.05
        )

        x = x + intensity_shift


    # --------------------------------------------------------
    # 4. Mild gamma augmentation
    # --------------------------------------------------------

    if random.random() < 0.5:

        gamma = random.uniform(
            0.90,
            1.10
        )

        x = torch.clamp(
            x,
            0.0,
            1.0
        )

        x = x.pow(gamma)


    # --------------------------------------------------------
    # 5. Small Gaussian noise
    # --------------------------------------------------------

    if random.random() < 0.5:

        sigma = random.uniform(
            0.0,
            0.025
        )

        noise = (
            torch.randn_like(x)
            * sigma
        )

        x = x + noise


    # --------------------------------------------------------
    # Final valid MRI range
    # --------------------------------------------------------

    x = torch.clamp(
        x,
        0.0,
        1.0
    )

    return x

In [ ]:
# ============================================================
# CLEAN STEP 7B
# Generate two independent views
# ============================================================

def make_ssl_views(x):

    view1 = mri_ssl_augment(x)
    view2 = mri_ssl_augment(x)

    return view1, view2


print("✓ MRI-safe two-view augmentation ready")

In [ ]:
test_row = ssl_triplets_clean.iloc[0]

series_uid = test_row["SeriesInstanceUID"]

percentile_row = (
    series_percentiles[
        series_percentiles["SeriesInstanceUID"]
        == series_uid
    ]
    .iloc[0]
)

p1 = float(percentile_row["P1"])
p99 = float(percentile_row["P99"])


base_sample = preprocess_triplet(
    test_row["PreviousPath"],
    test_row["CentrePath"],
    test_row["NextPath"],
    p1,
    p99
)

print("Base shape:", base_sample.shape)
print(
    "Base range:",
    float(base_sample.min()),
    float(base_sample.max())
)

In [ ]:
view1, view2 = make_ssl_views(
    base_sample
)

print("View 1 shape:", view1.shape)
print("View 2 shape:", view2.shape)

print(
    "View 1 range:",
    float(view1.min()),
    float(view1.max())
)

print(
    "View 2 range:",
    float(view2.min()),
    float(view2.max())
)

mean_abs_difference = (
    torch.mean(
        torch.abs(view1 - view2)
    )
    .item()
)

print(
    "Mean absolute difference:",
    mean_abs_difference
)

print(
    "Plane:",
    test_row["Anatomical_Plane"]
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    1,
    3,
    figsize=(12, 4)
)

axes[0].imshow(
    base_sample[1].cpu().numpy(),
    cmap="gray"
)
axes[0].set_title("Original")

axes[1].imshow(
    view1[1].cpu().numpy(),
    cmap="gray"
)
axes[1].set_title("SSL View 1")

axes[2].imshow(
    view2[1].cpu().numpy(),
    cmap="gray"
)
axes[2].set_title("SSL View 2")

for ax in axes:
    ax.axis("off")

plt.suptitle(
    f"MRI-safe SSL augmentation — "
    f"{test_row['Anatomical_Plane']}"
)

plt.tight_layout()
plt.show()